<a href="https://colab.research.google.com/github/carneiro-santos/dataton_fase_5_/blob/main/fase_5_FIAP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Datathon Fase 5

### nesta primeira etapa, tratamento dos dados

In [60]:
import pandas as pd
import numpy as np
import re
import unicodedata

pd.set_option('display.max_columns', None)

### carregamento da tabela

In [61]:
file_name = '/content/Tabela Geral Sheets-4.xlsx'

xls = pd.ExcelFile(file_name)

df_2022 = pd.read_excel(xls, sheet_name='2022')
df_2023 = pd.read_excel(xls, sheet_name='2023')
df_2024 = pd.read_excel(xls, sheet_name='2024')

print(df_2022.shape, df_2023.shape, df_2024.shape)

(860, 42) (1014, 50) (1156, 51)


### exploracao dos dados

In [62]:
for nome, df in zip(['2022','2023','2024'], [df_2022, df_2023, df_2024]):
    print(f"\n==== {nome} ====")
    print(df.columns.tolist())


==== 2022 ====
['RA', 'Fase', 'Turma', 'Nome', 'Ano nasc', 'Idade 22', 'Gênero', 'Ano ingresso', 'Instituição de ensino', 'Pedra 20', 'Pedra 21', 'Pedra 22', 'INDE 22', 'Cg', 'Cf', 'Ct', 'Nº Av', 'Avaliador1', 'Rec Av1', 'Avaliador2', 'Rec Av2', 'Avaliador3', 'Rec Av3', 'Avaliador4', 'Rec Av4', 'IAA', 'IEG', 'IPS', 'Rec Psicologia', 'IDA', 'Matem', 'Portug', 'Inglês', 'Indicado', 'Atingiu PV', 'IPV', 'IAN', 'Fase ideal', 'Defas', 'Destaque IEG', 'Destaque IDA', 'Destaque IPV']

==== 2023 ====
['RA', 'Fase', 'INDE 2023', 'Pedra 2023', 'Turma', 'Nome Anonimizado', 'Data de Nasc', 'Unnamed: 7', 'Idade', 'Unnamed: 9', 'Gênero', 'Ano ingresso', 'Instituição de ensino', 'Pedra 20', 'Pedra 21', 'Pedra 22', 'Pedra 23', 'INDE 22', 'INDE 23', 'Cg', 'Cf', 'Ct', 'Nº Av', 'Avaliador1', 'Rec Av1', 'Avaliador2', 'Rec Av2', 'Avaliador3', 'Rec Av3', 'Avaliador4', 'Rec Av4', 'IAA', 'IEG', 'IPS', 'IPP', 'Rec Psicologia', 'IDA', 'Mat', 'Por', 'Ing', 'Indicado', 'Atingiu PV', 'IPV', 'IAN', 'Fase Ideal', '

## Padronizacao das colunas

In [63]:
def remover_acentos(texto):
    return unicodedata.normalize('NFKD', str(texto)).encode('ASCII', 'ignore').decode('utf-8')

def ajustar_colunas(df, ano):

    df.columns = [remover_acentos(col).strip() for col in df.columns]

    rename_map = {}

    for col in df.columns:
        c = col.lower()

        if 'inde' in c: rename_map[col] = 'INDE'
        elif 'ida' in c: rename_map[col] = 'IDA'
        elif 'ieg' in c: rename_map[col] = 'IEG'
        elif 'iaa' in c: rename_map[col] = 'IAA'
        elif 'ips' in c: rename_map[col] = 'IPS'
        elif 'ipp' in c: rename_map[col] = 'IPP'
        elif 'ian' in c: rename_map[col] = 'IAN'
        elif 'ipv' in c: rename_map[col] = 'IPV'

        elif c == 'mat': rename_map[col] = 'Nota_Mat'
        elif c == 'por': rename_map[col] = 'Nota_Por'
        elif c == 'ing': rename_map[col] = 'Nota_Ing'

        elif 'pedra' in c: rename_map[col] = 'Pedra'
        elif 'genero' in c: rename_map[col] = 'Genero'
        elif 'idade' in c: rename_map[col] = 'Idade'
        elif 'instituicao' in c or 'escola' in c: rename_map[col] = 'Instituicao'
        elif 'ingresso' in c: rename_map[col] = 'Ano_Ingresso'
        elif c == 'ra': rename_map[col] = 'RA'
        elif 'fase' in c: rename_map[col] = 'Fase'

    df = df.rename(columns=rename_map)

    df = df.loc[:, ~df.columns.duplicated()]

    df['Ano_Base'] = ano

    return df

### aplicando a padronizacao na tabela

In [64]:
df_2022 = ajustar_colunas(df_2022, 2022)
df_2023 = ajustar_colunas(df_2023, 2023)
df_2024 = ajustar_colunas(df_2024, 2024)

### Padronizacao da fase

In [65]:
def padronizar_fase(valor):

    if pd.isna(valor):
        return np.nan

    v = str(valor).strip().upper()

    if v in ['ALFA','FASE 1','FASE 2','FASE 3','FASE 4',
             'FASE 5','FASE 6','FASE 7','FASE 8']:
        return v

    match = re.search(r'\d+', v)

    if match:
        num = int(match.group())

        if num == 0:
            return 'ALFA'
        elif 1 <= num <= 8:
            return f'FASE {num}'
        else:
            return 'FASE 8'

    return v

for df in [df_2022, df_2023, df_2024]:
    if 'Fase' in df.columns:
        df['Fase'] = df['Fase'].apply(padronizar_fase)

### concatenacao

In [66]:
df_full = pd.concat([df_2022, df_2023, df_2024], ignore_index=True, sort=False)

print(df_full.shape)

(3030, 49)


## Convertendo tipo de dados

In [67]:
cols_num = [
    'INDE','IAN','IDA','IEG','IAA','IPS','IPP','IPV',
    'Nota_Mat','Nota_Por','Nota_Ing','Idade'
]

for col in cols_num:
    if col in df_full.columns:
        df_full[col] = (
            df_full[col].astype(str)
            .str.replace(',', '.', regex=False)
        )
        df_full[col] = pd.to_numeric(df_full[col], errors='coerce')

## Limpeza de categorias

In [68]:
# Pedra
df_full['Pedra'] = df_full['Pedra'].astype(str).str.capitalize()

# Gênero
df_full['Genero'] = df_full['Genero'].astype(str).str.capitalize()

genero_fix = {'Menino':'Masculino','Menina':'Feminino'}
df_full['Genero'] = df_full['Genero'].replace(genero_fix)

# FEATURE ENGINEERING

In [69]:
# Tempo no programa
df_full['Tempo_Programa'] = df_full['Ano_Base'] - df_full['Ano_Ingresso']

# Pedra numérica
mapa = {'Quartzo':1,'Ágata':2,'Ametista':3,'Topázio':4}
df_full['Pedra_Num'] = df_full['Pedra'].map(mapa)

# Fase numérica
def fase_num(x):
    if pd.isna(x): return np.nan
    if 'ALFA' in x: return 0
    return int(x.replace('FASE ', ''))

df_full['Fase_Num'] = df_full['Fase'].apply(fase_num)

# Target
df_full['Risco'] = (df_full['IAN'] < 10).astype(int)

# Validacao

In [70]:
print(df_full.shape)

print("\nAnos:")
print(df_full['Ano_Base'].value_counts())

print("\nMissing:")
print(df_full.isnull().mean().sort_values(ascending=False).head(10))

print("\nNotas 2024:")
display(df_full[df_full['Ano_Base']==2024][[
    'Nota_Mat','Nota_Por','Nota_Ing'
]].head())

(3030, 53)

Anos:
Ano_Base
2024    1156
2023    1014
2022     860
Name: count, dtype: int64

Missing:
Avaliador6    0.998020
Avaliador5    0.951155
Ingles        0.906601
Rec Av4       0.902310
Nota_Ing      0.733333
Matem         0.716832
Portug        0.716832
Rec Av2       0.716172
Ano nasc      0.716172
Nome          0.716172
dtype: float64

Notas 2024:


,Nota_Mat,Nota_Por,Nota_Ing
1874,10.0,6.0,NaN
1875,10.0,6.0,NaN
1876,10.0,6.0,NaN
1877,8.0,6.0,NaN
1878,8.0,7.0,NaN


# Exportacao da base

In [71]:
df_full.to_excel('/content/base_final.xlsx', index=False)